# Week 10: Object Detection

**Lecture 16 — October 20:** Object Detection I  
**Lecture 17 — October 22:** Object Detection II

Image classification assigns one label to an entire image. Object detection must determine **what** objects are present and **where** each object appears. A detector predicts a variable-size collection of class labels, confidence scores, and bounding boxes.

## Learning goals

By the end of Week 10, you should be able to:

- represent and convert bounding boxes;
- calculate intersection over union;
- distinguish localization, objectness, classification, and confidence;
- explain confidence thresholds and non-maximum suppression;
- interpret precision-recall curves and mean average precision;
- describe the progression from R-CNN through Faster R-CNN; and
- explain the main components of a YOLO-style detector.

## Bounding boxes and IoU

Boxes commonly use corner coordinates $b=(x_{min},y_{min},x_{max},y_{max})$ or center-size coordinates $b=(x_{center},y_{center},w,h)$. Coordinates may be pixels or values normalized to $[0,1]$. The format and scale must always be explicit.

Intersection over union measures box overlap:

$$\operatorname{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}.$$

IoU is used to match predictions with targets, assign positive and negative candidates, remove duplicate predictions, and define evaluation thresholds such as AP@0.50 and AP@0.75. The implementation will include box conversion, clipping, area calculation, pairwise IoU, and edge-case tests.

## Confidence, NMS, and evaluation

A detector often proposes several overlapping boxes for one object. Non-maximum suppression keeps the highest-scoring candidate and removes lower-scoring boxes of the same class when their IoU exceeds a threshold. A low confidence threshold increases recall but admits more false positives; an aggressive NMS threshold can suppress nearby distinct objects.

A prediction is a true positive only when its class is correct and its IoU with an unmatched target exceeds the evaluation threshold.

$$\text{precision}=\frac{TP}{TP+FP}, \qquad \text{recall}=\frac{TP}{TP+FN}.$$

Average precision summarizes a class's precision-recall curve. Mean average precision averages across classes, and COCO-style mAP also averages across IoU thresholds from 0.50 through 0.95.

# Lecture 16: A Scratch Detector and the R-CNN Family

## Small detector from scratch

The first implementation will keep every stage visible: generate windows at several positions and scales, classify each crop, map window locations back to image coordinates, reject low-confidence candidates, and apply NMS. This exposes the detection pipeline but repeats convolution for overlapping crops and quickly becomes prohibitively expensive.

## R-CNN → Fast R-CNN → Faster R-CNN

- **R-CNN** replaces exhaustive windows with region proposals but runs a CNN separately on every proposed crop.
- **Fast R-CNN** runs the backbone once, then extracts fixed-size representations from a shared feature map using ROI pooling.
- **Faster R-CNN** adds a trainable region proposal network, using anchors to predict objectness and box offsets before second-stage classification and refinement.

Faster R-CNN is a two-stage detector: propose regions first, then classify and refine them. The full experiment will use torchvision's pretrained Faster R-CNN rather than reproduce the complete production architecture from scratch.

## Multi-task detection losses

A representative detection objective is

$$L=\lambda_{obj}L_{obj}+\lambda_{cls}L_{cls}+\lambda_{box}L_{box}.$$

Objectness separates object candidates from background, classification assigns categories, and box regression improves localization. These terms have different scales and numbers of contributing examples, so matching rules, class imbalance, and loss weights materially affect training.

# Lecture 17: One-stage Detection and YOLO

Two-stage detectors refine a smaller set of proposed regions. One-stage detectors predict dense boxes and class scores directly from feature maps. They are typically faster but must handle severe imbalance between background locations and objects. SSD predicts at multiple resolutions, while RetinaNet uses focal loss to reduce the contribution of easy background examples.

## A YOLO-style detector

A YOLO-style head assigns prediction to a spatial grid. Each location predicts box coordinates, objectness, and class scores. The planned implementation will:

1. encode targets into grid locations and box parameters;
2. compute image features with a pretrained backbone;
3. predict dense box, objectness, and class tensors;
4. calculate localization, objectness, and classification losses;
5. decode predictions into image coordinates; and
6. filter by confidence and apply class-aware NMS.

The goal is to implement the transferable mechanics directly, not reproduce an entire production YOLO framework.

## Detection-family comparison

| Approach | Candidate mechanism | Shared features | Main strength | Main limitation |
|---|---|---|---|---|
| Sliding windows | Exhaustive crops | No | Transparent baseline | Extremely expensive |
| R-CNN | External proposals | No | Better candidates | Repeated CNN computation |
| Fast R-CNN | External proposals | Yes | Shared feature map | Proposal bottleneck remains |
| Faster R-CNN | Learned proposals | Yes | Strong two-stage baseline | More complex and slower |
| YOLO-style | Dense grid predictions | Yes | Fast end-to-end detection | Dense matching and imbalance |

## Planned code components

- Bounding-box conversion, clipping, and pairwise IoU
- Class-aware non-maximum suppression
- Detection dataset and custom `collate_fn`
- Bounding-box visualization
- Scratch sliding-window baseline
- Torchvision Faster R-CNN inference and fine-tuning
- YOLO-style target encoding and prediction head
- Objectness, classification, and box losses
- Prediction decoding and confidence filtering
- Precision-recall and mAP evaluation

Executable code will be added after the dataset and experiment size are finalized.

## Preliminary takeaways

- Detection combines classification with localization and duplicate removal.
- IoU connects training assignments, NMS, and evaluation.
- R-CNN-family improvements progressively share computation and learn proposals.
- Faster R-CNN separates proposal generation from final classification and refinement.
- YOLO-style models predict dense boxes, objectness, and classes in one network.
- Detection metrics depend on confidence ranking and localization quality, not only class correctness.